# Lab 15 · The dialogue manager: the model understands, the graph controls

**Time:** 45 min · **Work in teams of 3** · **You will:** work with a complete **dialogue manager** built as a LangGraph graph, in which a language model sits in two nodes (`understand` and `answer_manual`) and everything else is code. You will run the **six conversation patterns** of Day 2's slides, read the decision at the heart of the graph (`route`), **break it on purpose**, and **measure the model against keyword rules** on the same conversations.

**The pipeline, one turn:** `understand` (model: message to intent and details) → `track` (code: update the task in progress) → `route` (code: decide) → one speaking node: `ask_slot`, `clarify`, `answer_manual` (model), `confirm`, `execute`, `cancel`, `fallback`.

The code of the manager is in `home_manager.py`. Open it in a second tab and read the nodes while you work.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**. Predict before each conversation.*

In [ ]:
import sys, os
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import check, Timer
import graph_slm
import home_manager as hm
import home_domain as hd
import inspect
import pandas as pd
from graph_viz import draw_graph
print("model:", graph_slm.MODEL)

## 13.1 The graph and its state
The **state** is the assistant's notebook: the task in progress, the home, whether we are waiting for a yes, how many times in a row we did not understand. It is saved by the checkpointer, one notebook per conversation.

In [ ]:
app = hm.build()                                    # the model reads the messages
draw_graph(app, "The dialogue manager")
print("state keys:", list(hm.DialogueState.__annotations__))

## 13.2 The decision at the heart: `route`
`route` asks its questions **in order**. The first that fits decides what the assistant does. **The order is the behaviour.** Read it:

In [ ]:
print(inspect.getsource(hm.route))

## 13.3 Six conversation patterns, one at a time
Each cell sends a short conversation into the graph and prints, for every turn, the reply, the **path** (which nodes ran) and the **task** in progress. **Predict first:** what will each reply be?

### Pattern 1: slot filling, and pattern 2: context
The user gives details a little at a time. The assistant asks only for what is missing.

In [ ]:
_ = hm.converse(app, ["set the ac", "bedroom", "22"], thread="p1")                  # 🔧 try your own order of details

### Pattern 3: ambiguity
Two lights are on. The user says "turn the light off". Which one? The assistant must **ask**, not guess. With only one light on, it can work it out.

In [ ]:
_ = hm.converse(app, ["turn on the kitchen light", "turn on the study light", "turn the light off", "the study"], thread="p3")
print()
_ = hm.converse(app, ["turn on the study light", "turn the light off"], thread="p3b")

### Pattern 4: multi-turn collection, and pattern 5: corrections
A later value **replaces** an earlier one. Nothing needs a special command: `track` merges new details over old.

In [ ]:
_ = hm.converse(app, ["set the ac to 24", "actually 20", "kitchen"], thread="p5")

### Pattern 6: interruption
In the middle of a task the user asks a question. The assistant **answers it with the model (from the manual), keeps the task, and carries on** with it.

In [ ]:
replies, state = hm.converse(app, ["set the bedroom ac", "what does E47 mean on the washer?", "21"], thread="p6")
print("\nfinal home:", hd.describe(state["home"]))

### Confirmation and failure
A risky action (the scene "away" switches everything off and locks up) needs a **yes**. Two messages in a row that are not understood bring up a **menu**.

In [ ]:
_ = hm.converse(app, ["i am leaving", "yes"], thread="c1")
print()
_ = hm.converse(app, ["blah blah", "more blah"], thread="c2")

## 13.4 Run the scenario tests
Eleven conversations, each with the **exact** replies expected. We run them against the **keyword rules** and against the **language model** in the same graph. Nothing else differs between the two: same `track`, same `route`, same speaking nodes.

In [ ]:
with Timer() as t_rules:
    passed_r, total, fails_r = hm.run_scenarios(lambda: hm.build(understand=hm.rules_understand), show=False)
with Timer() as t_slm:
    passed_m, _, fails_m = hm.run_scenarios(lambda: hm.build(understand=hm.slm_understand), show=False)
print(f"rules     : {passed_r} of {total} scenarios pass  ({t_rules.seconds:.1f} s)")
print(f"language model: {passed_m} of {total} scenarios pass  ({t_slm.seconds:.1f} s)")
for name, turns, expected, got in fails_m:
    print("FAIL (model):", name, "\n   turns   :", turns, "\n   expected:", expected, "\n   got     :", got)

In [ ]:
check(passed_r == total and passed_m >= total - 2,
      f"The rules pass all {total}, the model passes {passed_m}. The control logic is the same, so the scenarios test the **understanding**.",
      "rules should pass all scenarios; the model should pass nearly all. Read the FAIL lines.")

## 13.5 The same graph, messages the rules were never written for
People do not write "set_light". Five messages in everyday words: *"it is boiling in the bedroom, make it 22 degrees"*, *"can you kill the lamp in the study"*, *"turn on the livin room lite"* (a speech-recognition mistake), and two more. Same graph, same test, two understanders.

In [ ]:
pr, total2, fails_pr = hm.run_scenarios(lambda: hm.build(understand=hm.rules_understand), hm.PARAPHRASES, show=False)
pm, _, fails_pm = hm.run_scenarios(lambda: hm.build(understand=hm.slm_understand), hm.PARAPHRASES, show=False)
print(f"everyday wording: rules pass {pr} of {total2}, language model passes {pm} of {total2}")
for name, turns, expected, got in fails_pm:
    print("model FAIL:", name, "|", turns, "→", got)

In [ ]:
check(pm > pr, f"The model understood {pm - pr} more everyday messages than the rules. This is the reason a model sits in `understand`.",
      "the model should handle more of the everyday messages than the keyword rules.")

## 13.6 What the model's answer looks like before the code checks it
The `understand` node returns the model's raw text, whether it was valid JSON, and the values the code **dropped** because the home does not know them.

In [ ]:
for text in ["set the study ac to 45 degrees", "put the light on in the garage", "make the kitchen a bit cooler"]:
    out = hm.slm_understand({"utterance": text})
    n = out["nlu"]
    print(f"{text!r}\n   model wrote: {n['raw']}\n   intent: {n['intent']}  slots kept: {n['slots']}  dropped by the code: {n['dropped']}")

**Read it:** "45 degrees" and "the garage" are **dropped**. The model's answer is a *proposal*; only values that survive `clean_slots` reach the graph. A dropped slot simply becomes a missing slot, so the assistant **asks** instead of acting.

## 13.7 Break it on purpose
`route` checks "is the message ambiguous?" **before** "is a detail missing?". What if the order were the other way round? Build the graph with a changed routing function and rerun the scenarios.

In [ ]:
def route_wrong(state):
    task = state.get("task")
    if task and hm.missing(task) and state["nlu"]["intent"] != "ask_manual":      # 🔧 "ask for the missing detail" moved above "clarify"
        return "ask_slot"
    return hm.route(state)

pw, _, fails_w = hm.run_scenarios(lambda: hm.build(understand=hm.rules_understand, route_fn=route_wrong), show=False)
print(f"with the changed route: {pw} of {total} scenarios pass")
for name, turns, expected, got in fails_w: print("  broke:", name, "→ got", got)

In [ ]:
check(pw < total, "Moving one question changed the behaviour of a whole conversation pattern. That is why every behaviour has a test, and why `route` is read as a list of questions.",
      "the changed order should break at least one scenario.")

## 13.8 The cost of the model in the graph
One turn calls the model once (`understand`), twice if the user asks a question (`answer_manual`). Rules cost nothing. Add the numbers.

In [ ]:
per_turn_rules = t_rules.seconds / sum(len(t) for _, t, _ in hm.SCENARIOS)
per_turn_model = t_slm.seconds / sum(len(t) for _, t, _ in hm.SCENARIOS)
print(f"seconds per turn: rules {per_turn_rules:.3f} · language model {per_turn_model:.3f}")

**Read it:** the model costs a little time on every turn and buys understanding of everyday language. Day 3 is about making that time smaller.

## Check your understanding
1. Name the two nodes in which the model is used, and three nodes that are plain code.
*✍️ Your answer:* 
2. In the pattern "turn the light off" with two lights on, which node asked the question and what in the state made `route` choose it?
*✍️ Your answer:* 
3. The model proposed `temp: 45`. What did the code do, and what did the assistant ask next?
*✍️ Your answer:* 
4. Why does the same `track` and `route` make the comparison between rules and model fair?
*✍️ Your answer:* 

## Where this connects
The confirmation in this lab is a flag in the state ("awaiting yes"), checked at the next turn. In Lab 12 you saw the proper way: the graph **pauses** with `interrupt` until a human answers, and can survive a restart.

## Next
Day 3 starts with Lab 16: how to make a model like this fast, small and cheap to serve.